# Step 2 — Parse Search Logs and Build Search Results Table

## Purpose
Read all per-city txt files from `data_raw/311_search_log/`, parse the structured fields,
and produce `data_clean/search_results.csv` — one row per city with adoption year, source, and status.

## Input
`data_raw/311_search_log/*.txt` — one file per city, format:
```
city: [Name]
state: [2-letter code]
adoption_year: [YYYY or 'unknown' or 'no_311']
source_url: [URL]
source_type: [official / news / secondary]
evidence_note: [One sentence]
```

## Output
`data_clean/search_results.csv`

In [1]:
import re
import pandas as pd
from pathlib import Path

RAW   = '../data_raw/'
CLEAN = '../data_clean/'
LOG_DIR = Path(RAW) / '311_search_log'

## 1. Parse all txt files

In [2]:
def parse_txt(path):
    """Parse a search log txt file into a dict."""
    fields = {'filename': path.name}
    for line in path.read_text(encoding='utf-8').splitlines():
        if ':' in line:
            key, _, val = line.partition(':')
            key = key.strip()
            val = val.strip()
            # evidence_note may contain colons — take everything after first colon
            if key == 'evidence_note':
                val = line.partition(':')[2].strip()
            if key == 'source_url':
                # source_url contains http:// — rejoin after first colon
                val = line.partition(':')[2].strip()
            fields[key] = val
    return fields

records = []
for p in sorted(LOG_DIR.glob('*.txt')):
    d = parse_txt(p)
    records.append(d)

raw_df = pd.DataFrame(records)
print(f'Parsed {len(raw_df)} txt files')
print(f'Columns: {list(raw_df.columns)}')

Parsed 316 txt files
Columns: ['filename', 'city', 'state', 'adoption_year', 'source_url', 'source_type', 'evidence_note']


## 2. Normalise adoption_year field

In [3]:
def parse_year(val):
    """Return integer year, or None for unknown/no_311/missing."""
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if s in ('unknown', 'no_311', 'none', ''):
        return None
    m = re.search(r'\b(19|20)\d{2}\b', s)
    return int(m.group()) if m else None

raw_df['year_found'] = raw_df['adoption_year'].apply(parse_year)

# Normalise adoption_year to clean string
def norm_status(val):
    if pd.isna(val):
        return 'unknown'
    s = str(val).strip().lower()
    if s == 'no_311':
        return 'no_311'
    if parse_year(val) is not None:
        return str(parse_year(val))
    return 'unknown'

raw_df['adoption_year_clean'] = raw_df['adoption_year'].apply(norm_status)

print('adoption_year_clean distribution:')
year_dist = raw_df['adoption_year_clean'].value_counts()
print(f'  Years found : {(raw_df["year_found"].notna()).sum()}')
print(f'  unknown     : {(raw_df["adoption_year_clean"] == "unknown").sum()}')
print(f'  no_311      : {(raw_df["adoption_year_clean"] == "no_311").sum()}')

adoption_year_clean distribution:
  Years found : 121
  unknown     : 140
  no_311      : 55


## 3. Match to city_list on name + state to get GEOID

In [4]:
from difflib import get_close_matches

city_list = pd.read_csv(CLEAN + 'city_list.csv', dtype={'GEOID': str})

SUFFIXES = re.compile(
    r'\b(city|town|village|borough|municipality|county|metro(politan)?'
    r'|government|consolidated|unified|urban|charter)\b',
    re.IGNORECASE
)

def norm(name):
    name = SUFFIXES.sub('', str(name))
    name = re.sub(r'[^a-z\s]', '', name.lower())
    return re.sub(r'\s+', ' ', name).strip()

city_list['name_norm'] = city_list['place_name'].apply(norm)
raw_df['name_norm']    = raw_df['city'].apply(norm)

lookup = (
    city_list.groupby(['state_abbr', 'name_norm'])['GEOID']
    .first().reset_index()
    .set_index(['state_abbr', 'name_norm'])['GEOID'].to_dict()
)
state_names = city_list.groupby('state_abbr')['name_norm'].apply(list).to_dict()

def match(row):
    key = (row['state'], row['name_norm'])
    if key in lookup:
        return lookup[key], 'exact'
    cands = state_names.get(row['state'], [])
    m = get_close_matches(row['name_norm'], cands, n=1, cutoff=0.82)
    if m:
        return lookup[(row['state'], m[0])], f'fuzzy:{m[0]}'
    return None, 'no_match'

raw_df[['GEOID', 'match_type']] = raw_df.apply(match, axis=1, result_type='expand')

print('Match summary:')
print(raw_df['match_type'].str.split(':').str[0].value_counts().to_string())
unmatched = raw_df[raw_df['GEOID'].isna()]
if len(unmatched):
    print(f'\nUnmatched ({len(unmatched)}):')
    print(unmatched[['city', 'state', 'filename']].to_string(index=False))

Match summary:
match_type
exact    316


## 4. Merge with city_list; add missing cities as 'not_searched'

In [5]:
# Keep best match per GEOID (earliest year if multiple log files hit same city)
matched = (
    raw_df[raw_df['GEOID'].notna()]
    .sort_values('year_found', na_position='last')
    .drop_duplicates('GEOID', keep='first')
    [['GEOID', 'city', 'state', 'adoption_year_clean', 'year_found',
      'source_url', 'source_type', 'evidence_note', 'match_type']]
    .rename(columns={'adoption_year_clean': 'adoption_year_raw'})
)

results = city_list[['GEOID', 'place_name', 'state_abbr', 'pop_2020']].merge(
    matched, on='GEOID', how='left'
)

# Cities not in any log file
not_searched = results['adoption_year_raw'].isna() & results['city'].isna()
results.loc[not_searched, 'adoption_year_raw'] = 'not_searched'

print(f'Total cities in sample : {len(results)}')
print(f'Log files matched      : {matched["GEOID"].nunique()}')
print(f'Not yet searched       : {not_searched.sum()}')
print()
print('Breakdown:')
print(results['adoption_year_raw'].apply(
    lambda x: 'year_found' if (str(x).isdigit()) else x
).value_counts().to_string())

Total cities in sample : 314
Log files matched      : 314
Not yet searched       : 0

Breakdown:
adoption_year_raw
unknown       138
year_found    121
no_311         55


## 5. Save

In [6]:
col_order = [
    'GEOID', 'place_name', 'state_abbr', 'pop_2020',
    'adoption_year_raw', 'year_found',
    'source_url', 'source_type', 'evidence_note', 'match_type'
]
out = results[col_order].copy()

out_path = CLEAN + 'search_results.csv'
out.to_csv(out_path, index=False)
print(f'Saved {len(out)} rows → {out_path}')
print()
print('Cities with adoption year (sorted by year):')
with_year = out[out['year_found'].notna()].sort_values('year_found')
print(with_year[['year_found', 'place_name', 'state_abbr', 'source_type']].to_string(index=False))

Saved 314 rows → ../data_clean/search_results.csv

Cities with adoption year (sorted by year):
 year_found             place_name state_abbr source_type
     1996.0         Baltimore city         MD    official
     1999.0           Chicago city         IL    official
     1999.0           Houston city         TX    official
     1999.0         Rochester city         NY    official
     1999.0        Birmingham city         AL    official
     2000.0         Las Vegas city         NV        news
     2000.0       San Antonio city         TX    official
     2000.0           Buffalo city         NY    official
     2000.0            Austin city         TX    official
     2002.0       Los Angeles city         CA    official
     2002.0            Dallas city         TX    official
     2003.0       Minneapolis city         MN    official
     2003.0       Chattanooga city         TN        news
     2003.0          New York city         NY    official
     2003.0          Columbus city 